# Two qubits
**Quantum Computing · Week 4 (Entanglement) · Day 1**

Lesson page: https://sara-aminpour.com/tutorials/qc/w4d1.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

Two qubits have **four** amplitudes, one for each of |00⟩, |01⟩, |10⟩, |11⟩.
Today: Qiskit's bit order (a classic gotcha), tensor products, and how fast the numbers grow.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

def probs(sv):
    """Probabilities as a tidy dictionary, skipping zeros."""
    return {str(k): round(float(v), 3) for k, v in sv.probabilities_dict().items()}

for label in ["00", "01", "10", "11"]:
    sv = Statevector.from_label(label)
    print(f"|{label}>  ->  amplitude vector {sv.data.real.astype(int)}")

## The bit-order gotcha
Qiskit writes bitstrings as **q1 q0**: qubit 0 is the *rightmost* character.
Flip only qubit 0 and you get the string `01`, not `10`.

In [ ]:
qc = QuantumCircuit(2)
qc.x(0)                      # flip qubit 0 only
print("after X on qubit 0:", probs(Statevector(qc)))

qc = QuantumCircuit(2)
qc.x(1)                      # flip qubit 1 only
print("after X on qubit 1:", probs(Statevector(qc)))

## Tensor products
Two independent qubits combine with the **tensor product** $\otimes$: multiply every amplitude of one
by every amplitude of the other. In Qiskit, `b.tensor(a)` puts `a` on qubit 0 and `b` on qubit 1.

In [ ]:
a = Statevector.from_label("+")      # will be qubit 0
b = Statevector.from_label("0")      # will be qubit 1
both = b.tensor(a)
print("amplitudes of |0> ⊗ |+> :", np.round(both.data.real, 3))
print("probabilities           :", probs(both))

qc = QuantumCircuit(2)
qc.h(0)                              # the same state, built as a circuit
print("same from a circuit     :", probs(Statevector(qc)))

For independent qubits, probabilities simply **multiply**:

In [ ]:
qc = QuantumCircuit(2)
qc.ry(1.2, 0)
qc.ry(2.0, 1)
p = probs(Statevector(qc))
print(p)

q0 = [np.cos(1.2 / 2) ** 2, np.sin(1.2 / 2) ** 2]     # qubit 0 alone: P(0), P(1)
q1 = [np.cos(2.0 / 2) ** 2, np.sin(2.0 / 2) ** 2]     # qubit 1 alone
print("P(q1 = 1 and q0 = 0) =", round(q1[1] * q0[0], 3), "  (compare with '10' above)")

## How fast does it grow?
Every extra qubit doubles the number of amplitudes a classical computer must store.

In [ ]:
def human(nbytes):
    for unit in ["bytes", "KB", "MB", "GB", "TB", "PB"]:
        if nbytes < 1024:
            return f"{nbytes:,.0f} {unit}"
        nbytes /= 1024
    return f"{nbytes:,.0f} EB"

for n in [1, 2, 3, 10, 20, 30, 40, 50]:
    amplitudes = 2 ** n
    print(f"{n:2d} qubits -> {amplitudes:>25,d} amplitudes = {human(amplitudes * 16):>9}")   # 16 bytes each

## Your turn: is it a product?
A two-qubit state $(v_{00}, v_{01}, v_{10}, v_{11})$ is a product of two one-qubit states exactly when
$v_{00}v_{11} - v_{01}v_{10} = 0$. Write `is_product` and test it on three states.

In [ ]:
def is_product(v):
    v = np.asarray(v, dtype=complex)
    return None      # TODO: True when |v[0]*v[3] - v[1]*v[2]| is (almost) zero

s = 1 / np.sqrt(2)
tests = {"|00>": [1, 0, 0, 0], "|+0> (q1 = +, q0 = 0)": [s, 0, s, 0], "(|00> + |11>)/sqrt2": [s, 0, 0, s]}
for name, v in tests.items():
    print(f"{name:>22}: product? {is_product(v)}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def is_product(v):
    v = np.asarray(v, dtype=complex)
    return bool(abs(v[0] * v[3] - v[1] * v[2]) < 1e-9)

for name, v in tests.items():
    print(f"{name:>22}: product? {is_product(v)}")
print("The last one can't be written as (one qubit) x (other qubit): it is entangled. Tomorrow's lesson makes it.")